# 🚀 TrafForesight-AI Analysis
### Intelligent Traffic Prediction & Congestion Estimation

This notebook provides a comprehensive analysis of the TrafForesight-AI project, including data preprocessing, model training, baseline comparison, and advanced visualizations.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import os
import sys

# Add model directory to path to import DataPreprocessor
sys.path.append('../model')
from preprocess import DataPreprocessor

sns.set_theme(style="darkgrid")

## 2. Load Dataset
The dataset contains simulated traffic volume records with timestamped features.

In [ ]:
df = pd.read_csv('../data/traffic.csv')
print(f"Dataset loaded with {len(df)} records.")
df.head()

## 3. Data Preprocessing
Using our custom `DataPreprocessor` to handle missing values and extract peak-hour features.

In [ ]:
preprocessor = DataPreprocessor()
df_processed = preprocessor.fit_transform(df, features=['speed'])

X = df_processed[['day_of_week', 'hour', 'weather', 'speed', 'is_weekend', 'is_peak_hour']]
y = df_processed['vehicle_count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print("Splitting complete: 80% Train, 20% Test.")

## 4. Model Training & Evaluation
Comparing Random Forest against Linear Regression and a Baseline (Mean).

In [ ]:
# Models
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
preds_rf = model_rf.predict(X_test)

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
preds_lr = model_lr.predict(X_test)

baseline_pred = np.full_like(y_test, y_train.mean())

# Metrics
def get_metrics(true, pred, name):
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    return {"Model": name, "MAE": mae, "RMSE": rmse}

results = [
    get_metrics(y_test, baseline_pred, "Baseline (Mean)"),
    get_metrics(y_test, preds_lr, "Linear Regression"),
    get_metrics(y_test, preds_rf, "Random Forest")
]

pd.DataFrame(results)

## 5. Visualizations

### 5.1 Time-Series Forecast (Actual vs Predicted)

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(y_test.values[:120], label='Actual', alpha=0.8, color='blue', linewidth=2)
plt.plot(preds_rf[:120], label='RF Prediction', linestyle='--', color='orange', linewidth=2)
plt.title('Traffic Volume Forecasting - Next 5 Days (120 Hours)', fontsize=15)
plt.xlabel('Hours', fontsize=12)
plt.ylabel('Vehicle Count', fontsize=12)
plt.legend()
plt.show()

### 5.2 Feature Importance

In [ ]:
plt.figure(figsize=(10, 5))
importance = model_rf.feature_importances_
features = X.columns
sns.barplot(x=importance, y=features, palette='magma')
plt.title('Drivers of Traffic Volume (Feature Importance)', fontsize=15)
plt.show()

### 5.3 Congestion Heatmap

In [ ]:
plt.figure(figsize=(12, 6))
heatmap_data = df.pivot_table(values='vehicle_count', index='day_of_week', columns='hour', aggfunc='mean')
sns.heatmap(heatmap_data, cmap="YlOrRd", annot=False)
plt.title('Hourly Congestion Patterns by Weekday', fontsize=15)
plt.ylabel('Day of Week (0=Mon)', fontsize=12)
plt.xlabel('Hour of Day', fontsize=12)
plt.show()

## 6. Conclusion
The Random Forest model effectively captures hourly and daily traffic fluctuations, significantly outperforming the standard baseline. The model is now ready for deployment via the FastAPI service.